# mcp-ogc — tool walkthrough

This notebook demonstrates the three `mcp-ogc` tools by calling them **directly** (no MCP client, no API key needed). It shows what an LLM agent receives when it calls each tool.

**Prerequisites:** `uv sync` in the repo root, then run this notebook with the project's `.venv` kernel.

**Network note:** the cells below use public endpoints that should be reachable from most networks. If one does not resolve on your network, swap in any other WMS/WFS endpoint you can reach.

In [ ]:
from mcp_ogc.tools.wfs import query_wfs_features
from mcp_ogc.tools.wms import get_wms_map, list_wms_layers

## 1. `list_wms_layers` — discover what a WMS endpoint offers

We point at the German BKG *TopPlusOpen* WMS (a stable, worldwide public basemap) and list its layers.

In [ ]:
WMS_URL = "https://sgx.geodatenzentrum.de/wms_topplus_open"

layers = list_wms_layers(WMS_URL)
print(f"{len(layers)} layers found\n")
for layer in layers:
    print(f"- {layer.name}: {layer.title}")

In [ ]:
# Full metadata for the first layer (this is what the agent sees):
print(layers[0].model_dump_json(indent=2))

## 2. `get_wms_map` — render a map image for a bounding box

We fetch a map of central Germany. `get_wms_map` returns raw PNG bytes; here we display them inline.

In [ ]:
from IPython.display import Image as IPyImage

png_bytes = get_wms_map(
    WMS_URL,
    layer="web",
    bbox=(10.0, 50.0, 11.0, 51.0),
    crs="EPSG:4326",
    width=600,
    height=600,
)
print(f"{len(png_bytes)} bytes of PNG")
IPyImage(data=png_bytes)

## 3. `query_wfs_features` — query vector features as GeoJSON

We query a public WFS (Dutch national PDOK administrative-units service) for a few features. The tool returns a GeoJSON `FeatureCollection` dict.

_Swap in any reachable WFS + `type_name` from its capabilities if this endpoint is unavailable._

In [ ]:
WFS_URL = "https://service.pdok.nl/kadaster/bestuurlijkegebieden/wfs/v1_0"

fc = query_wfs_features(
    WFS_URL,
    type_name="bestuurlijkegebieden:Gemeentegebied",
    max_features=3,
)
print("type:", fc["type"])
print("feature count:", len(fc["features"]))
if fc["features"]:
    print("first feature property keys:", list(fc["features"][0]["properties"].keys()))

## That's it

Three tools: discover layers, render a map, query features — all against standards-compliant OGC services, all returning structured data an LLM agent can reason over. To use these from Claude Desktop or another MCP client, run `uv run mcp-ogc` and attach with the config in `examples/claude_desktop.json`.